# SciRet v2 — Notebook 02: Text Chunking Strategies

**Purpose:** Understand, implement, and compare different chunking strategies before committing to one for the full pipeline.

**Why a whole notebook just for chunking?**  
Chunking is one of the most impactful decisions in a RAG system — more than the choice of embedding model in many cases. A bad chunking strategy means:
- Relevant information split across chunk boundaries → retrieval misses it
- Chunks too large → embedding vector is diluted with noise
- Chunks too small → answer requires multiple chunks but only one is retrieved

**What we do:**
1. Understand what chunking is and why it matters mathematically
2. Implement four different chunking strategies
3. Compare them visually and quantitatively
4. Select the best strategy for SciRet v2 with justification
5. Save the final chunked dataset for Notebook 03

**Prerequisite:** Notebook 01 must have been run — we load `papers_clean.parquet`.

---
> **Research connection:** Your chunking choice will be reported in the System Architecture section of your paper. The comparison here gives you the evidence to justify that choice.

## Step 0 — Install Dependencies

In [ ]:
%pip install nltk matplotlib seaborn pandas numpy tqdm --quiet

import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)
print('✓ Dependencies ready')

## Step 1 — Imports

In [ ]:
import re
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
from dataclasses import dataclass, field
from typing import List, Dict, Callable
from nltk.tokenize import sent_tokenize

warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor': '#0d1117', 'axes.facecolor': '#0d1117',
    'axes.edgecolor':   '#334155', 'axes.labelcolor': '#94a3b8',
    'xtick.color':      '#64748b', 'ytick.color':     '#64748b',
    'text.color':       '#e2e8f0', 'grid.color':      '#1e293b',
    'grid.linestyle':   '--',      'font.family':     'monospace',
})
ACCENT, ACCENT2, ACCENT3, ACCENT4 = '#3b82f6', '#8b5cf6', '#10b981', '#f97316'

print('✓ Imports complete')

## Step 2 — Configuration

In [ ]:
ROOT_DIR  = Path('..').resolve()
DATA_PROC = ROOT_DIR / '1_data' / 'processed'

# How many papers to use for chunking comparison
# We only need a sample — chunking is fast, we just want representative statistics
N_SAMPLE = 100

# Chunking parameters to test
# We test multiple sizes to find the best trade-off
CHUNK_SIZES    = [200, 300, 400, 500]   # tokens
OVERLAP_SIZES  = [30,  50,  75,  100]   # tokens

# Final choice — we will justify this at the end
FINAL_CHUNK_SIZE    = 400
FINAL_OVERLAP       = 50

print(f'✓ Config set')
print(f'  Sample papers  : {N_SAMPLE}')
print(f'  Chunk sizes    : {CHUNK_SIZES}')
print(f'  Final choice   : {FINAL_CHUNK_SIZE} tokens, {FINAL_OVERLAP} overlap')

## Step 3 — Load Data from Notebook 01

In [ ]:
papers_path = DATA_PROC / 'papers_clean.parquet'
assert papers_path.exists(), f'Run Notebook 01 first. Missing: {papers_path}'

df = pd.read_parquet(papers_path).head(N_SAMPLE)

print(f'✓ Loaded {len(df)} papers')
print(f'  Columns: {list(df.columns)}')

# Build a full text field: title + abstract
# In a later notebook we will also use body text from JSON files
df['full_text'] = df['title_clean'].fillna('') + '. ' + df['abstract_clean'].fillna('')
df['word_count'] = df['full_text'].str.split().str.len()

print(f'\n── Text length stats ────────────────────────')
print(f'  Mean words : {df["word_count"].mean():.0f}')
print(f'  Min  words : {df["word_count"].min()}')
print(f'  Max  words : {df["word_count"].max()}')

## Step 4 — What Is Chunking and Why Does It Matter?

Before implementing anything, let us build intuition for what chunking does.

**The embedding model perspective:**  
When BGE-M3 embeds a piece of text, it produces one vector that represents the **average meaning** of the whole text. 

Imagine a paper abstract that discusses:
- paragraph 1: CT scan imaging methodology  
- paragraph 2: patient demographics  
- paragraph 3: statistical analysis results  

If you embed the whole abstract as one vector, the vector is pulled in all three directions. A query about CT scans will not score very highly against this averaged vector — the demographics and statistics dilute it.

If you chunk it into three pieces, the first chunk's vector points strongly toward CT imaging. A query about CT scans scores very highly against it.

**Mathematically:**  
The embedding of a long text is approximately the mean of the embeddings of its parts:  
`embed(A + B + C) ≈ mean(embed(A), embed(B), embed(C))`

This means the cosine similarity of a query to a long chunk is diluted compared to a focused short chunk. Chunking increases the signal-to-noise ratio of your retrieval.

In [ ]:
# Helper: fast token count estimate
# 1 token ≈ 0.75 words (English scientific text)
# Real tokeniser would be more accurate but much slower
def est_tokens(text: str) -> int:
    return max(1, int(len(str(text).split()) / 0.75))


# Visualise why chunking matters — show topic drift in a long text
sample_text = df['full_text'].iloc[0]
sentences   = sent_tokenize(sample_text)

print(f'Sample paper: "{df["title_clean"].iloc[0][:70]}"')
print(f'Sentences   : {len(sentences)}')
print(f'Total tokens: ~{est_tokens(sample_text)}')
print()
print('Sentence-by-sentence view:')
for i, s in enumerate(sentences[:8], 1):
    print(f'  S{i:02d} [{est_tokens(s):3d} tok] {s[:90]}')
if len(sentences) > 8:
    print(f'  ... ({len(sentences)-8} more sentences)')

## Step 5 — Implement the Four Chunking Strategies

We implement four strategies, each with increasing sophistication:

| Strategy | Description | Pros | Cons |
|----------|-------------|------|------|
| **Fixed character** | Split every N characters | Simplest, fastest | Cuts words and sentences mid-way |
| **Fixed token** | Split every N tokens (words) | Predictable size | Still cuts sentences mid-way |
| **Sentence window** | Accumulate sentences up to N tokens | Preserves sentence boundaries | Variable chunk size |
| **Paragraph** | Split on paragraph breaks first | Preserves logical structure | Very variable size — some paragraphs huge |

**We implement all four** so we can compare their output on the same paper.

In [ ]:
# ── Strategy 1: Fixed Character ──────────────────────────────────────────────
def chunk_fixed_char(
    text: str,
    chunk_chars: int = 1500,
    overlap_chars: int = 150
) -> List[str]:
    """
    Split text every chunk_chars characters with overlap.

    This is the simplest possible strategy.
    Problem: it blindly cuts in the middle of words and sentences.
    Example: '...COVID-19 affects the respirat' | 'ory system by...'
    """
    if not text or len(text) < 50:
        return []

    chunks = []
    start  = 0
    while start < len(text):
        end = min(start + chunk_chars, len(text))
        chunks.append(text[start:end].strip())
        start += chunk_chars - overlap_chars

    return [c for c in chunks if len(c) > 30]


# ── Strategy 2: Fixed Token ───────────────────────────────────────────────────
def chunk_fixed_token(
    text: str,
    chunk_tokens: int = 400,
    overlap_tokens: int = 50
) -> List[str]:
    """
    Split text every chunk_tokens words with overlap.

    Better than character split — preserves whole words.
    Problem: still cuts mid-sentence.
    Example: 'The virus enters' | 'cells by binding to the ACE2'
    """
    if not text or len(text.split()) < 20:
        return []

    words  = text.split()
    chunks = []
    start  = 0

    while start < len(words):
        end = min(start + chunk_tokens, len(words))
        chunks.append(' '.join(words[start:end]))
        start += chunk_tokens - overlap_tokens

    return [c for c in chunks if len(c.split()) > 10]


# ── Strategy 3: Sentence Window ───────────────────────────────────────────────
def chunk_sentence_window(
    text: str,
    chunk_tokens: int = 400,
    overlap_tokens: int = 50
) -> List[str]:
    """
    Accumulate sentences until token budget is reached, then start new chunk.
    Overlap: carry forward sentences from the end of the previous chunk.

    This is the best general-purpose strategy:
    - Never cuts mid-sentence
    - Overlap preserves context at boundaries
    - Handles variable sentence lengths naturally

    Algorithm:
    1. Tokenise text into sentences using NLTK
    2. Build a buffer of sentences
    3. When buffer exceeds chunk_tokens: save chunk, step back overlap_tokens
    4. Repeat
    """
    if not text or len(text.strip()) < 50:
        return []

    sentences   = sent_tokenize(text)
    chunks      = []
    buffer      = []
    buffer_toks = 0

    for sent in sentences:
        sent_toks = est_tokens(sent)

        # Single sentence exceeds budget — add alone
        if sent_toks > chunk_tokens:
            if buffer:
                chunks.append(' '.join(buffer))
                buffer, buffer_toks = [], 0
            chunks.append(sent)
            continue

        # Buffer would exceed budget — flush and start overlap
        if buffer_toks + sent_toks > chunk_tokens and buffer:
            chunks.append(' '.join(buffer))

            # Build overlap from end of buffer
            overlap_buf, overlap_toks = [], 0
            for s in reversed(buffer):
                s_toks = est_tokens(s)
                if overlap_toks + s_toks > overlap_tokens:
                    break
                overlap_buf.insert(0, s)
                overlap_toks += s_toks

            buffer, buffer_toks = overlap_buf, overlap_toks

        buffer.append(sent)
        buffer_toks += sent_toks

    if buffer:
        chunks.append(' '.join(buffer))

    return [c.strip() for c in chunks if len(c.strip()) > 30]


# ── Strategy 4: Paragraph-Based ───────────────────────────────────────────────
def chunk_paragraph(
    text: str,
    chunk_tokens: int = 400,
    overlap_tokens: int = 50
) -> List[str]:
    """
    Split on paragraph boundaries first, then apply sentence window
    if a paragraph is still too large.

    Respects the logical structure of scientific papers.
    Problem: abstracts often have no paragraph breaks — treated as one unit.
    Works better when you have full paper body text (JSON files).
    """
    if not text or len(text.strip()) < 50:
        return []

    # Split on double newlines (paragraph breaks)
    paragraphs = re.split(r'\n{2,}', text.strip())
    paragraphs = [p.strip() for p in paragraphs if p.strip()]

    chunks = []
    buffer, buffer_toks = [], 0

    for para in paragraphs:
        para_toks = est_tokens(para)

        # Paragraph itself is too large — apply sentence window to it
        if para_toks > chunk_tokens:
            if buffer:
                chunks.append('\n'.join(buffer))
                buffer, buffer_toks = [], 0
            sub_chunks = chunk_sentence_window(para, chunk_tokens, overlap_tokens)
            chunks.extend(sub_chunks)
            continue

        # Would exceed budget — flush buffer
        if buffer_toks + para_toks > chunk_tokens and buffer:
            chunks.append('\n'.join(buffer))
            buffer, buffer_toks = [], 0

        buffer.append(para)
        buffer_toks += para_toks

    if buffer:
        chunks.append('\n'.join(buffer))

    return [c.strip() for c in chunks if len(c.strip()) > 30]


print('✓ Four chunking strategies defined')
print('  1. chunk_fixed_char       — split every N characters')
print('  2. chunk_fixed_token      — split every N words')
print('  3. chunk_sentence_window  — accumulate sentences (our choice)')
print('  4. chunk_paragraph        — paragraph boundaries first')

## Step 6 — Visual Comparison on One Paper

Before running on the whole dataset, let us see what each strategy produces on a single paper.

**What to look for:**
- Does the chunk start and end at a natural boundary?
- Is the chunk size close to our target?
- Does the beginning of the next chunk make sense on its own?

In [ ]:
# Pick one paper with a reasonably long abstract
sample_idx  = df['word_count'].nlargest(5).index[0]
sample_text = df['full_text'].iloc[sample_idx]
sample_title = df['title_clean'].iloc[sample_idx]

print(f'Sample paper: "{sample_title[:70]}"')
print(f'Total tokens: ~{est_tokens(sample_text)}')
print(f'Total words : {len(sample_text.split())}')
print('=' * 70)

strategies = {
    'Fixed Character'     : chunk_fixed_char(sample_text, 1500, 150),
    'Fixed Token'         : chunk_fixed_token(sample_text, 400, 50),
    'Sentence Window'     : chunk_sentence_window(sample_text, 400, 50),
    'Paragraph'           : chunk_paragraph(sample_text, 400, 50),
}

for name, chunks in strategies.items():
    print(f'\n── {name} ──────────────────────────────────')
    print(f'   Produced {len(chunks)} chunks')
    print()
    for i, chunk in enumerate(chunks[:2], 1):   # show first 2 chunks
        preview = chunk[:200].replace('\n', ' ')
        print(f'   Chunk {i} ({est_tokens(chunk)} tokens):')
        print(f'   START: "{preview[:100]}"')
        print(f'   END  : "...{chunk[-80:].replace(chr(10), " ")}"')
        print()

## Step 7 — Boundary Quality Analysis

**The key quality indicator:** Does each chunk start with a capital letter and end with punctuation?

Fixed character/token splits frequently violate this. Sentence window and paragraph splits should preserve it almost always.

This matters because an embedding of `"ory system by binding to ACE2"` is worse than one starting at a sentence boundary — the model has no context for what `ory system` refers to.

In [ ]:
def analyse_boundary_quality(chunks: List[str]) -> Dict:
    """
    Measure how well chunks respect natural text boundaries.

    Metrics:
    - starts_capital : chunk begins with uppercase letter (sentence start)
    - ends_punct     : chunk ends with . ? ! (sentence end)
    - both_clean     : both conditions met (clean boundary)
    - avg_tokens     : average chunk size
    - std_tokens     : standard deviation of chunk sizes (lower = more consistent)
    - cv_tokens      : coefficient of variation = std/mean (lower = more consistent)
    """
    if not chunks:
        return {}

    starts_cap  = sum(1 for c in chunks if c and c[0].isupper()) / len(chunks)
    ends_punct  = sum(1 for c in chunks if c and c[-1] in '.!?"\'')
    ends_punct  = ends_punct / len(chunks)
    both_clean  = sum(
        1 for c in chunks
        if c and c[0].isupper() and c[-1] in '.!?"\'\''
    ) / len(chunks)

    token_sizes = [est_tokens(c) for c in chunks]
    avg_toks    = np.mean(token_sizes)
    std_toks    = np.std(token_sizes)

    return {
        'n_chunks'      : len(chunks),
        'starts_capital': round(starts_cap * 100, 1),
        'ends_punct'    : round(ends_punct * 100, 1),
        'both_clean'    : round(both_clean * 100, 1),
        'avg_tokens'    : round(avg_toks, 1),
        'std_tokens'    : round(std_toks, 1),
        'cv_tokens'     : round(std_toks / avg_toks * 100, 1) if avg_toks > 0 else 0,
    }


# Run on sample paper
print('── Boundary Quality on Sample Paper ──────────────────────────────')
print(f'{"Strategy":<22} {"Chunks":>6} {"Cap%":>6} {"Punct%":>7} {"Clean%":>7} {"AvgTok":>7} {"CV%":>6}')
print('─' * 68)

quality_results = {}
for name, chunks in strategies.items():
    q = analyse_boundary_quality(chunks)
    quality_results[name] = q
    print(
        f'{name:<22} {q["n_chunks"]:>6} '
        f'{q["starts_capital"]:>6.1f} '
        f'{q["ends_punct"]:>7.1f} '
        f'{q["both_clean"]:>7.1f} '
        f'{q["avg_tokens"]:>7.1f} '
        f'{q["cv_tokens"]:>6.1f}'
    )

print()
print('Cap%   = % chunks starting with capital letter (↑ better)')
print('Punct% = % chunks ending with punctuation     (↑ better)')
print('Clean% = % chunks with both clean boundaries  (↑ better)')
print('CV%    = coefficient of variation of sizes    (↓ better = more consistent)')

## Step 8 — Scale Analysis Across All Sample Papers

The single-paper comparison showed quality. Now we check consistency across all 100 papers.  
A good chunking strategy should behave predictably regardless of paper length or writing style.

In [ ]:
print(f'Running all 4 strategies on {len(df)} papers...')

strategy_fns = {
    'Fixed Character' : lambda t: chunk_fixed_char(t, 1500, 150),
    'Fixed Token'     : lambda t: chunk_fixed_token(t, 400, 50),
    'Sentence Window' : lambda t: chunk_sentence_window(t, 400, 50),
    'Paragraph'       : lambda t: chunk_paragraph(t, 400, 50),
}

scale_results = {name: [] for name in strategy_fns}

for _, row in tqdm(df.iterrows(), total=len(df), desc='Analysing'):
    text = row['full_text']
    for name, fn in strategy_fns.items():
        chunks = fn(text)
        q      = analyse_boundary_quality(chunks)
        q['paper_words'] = row['word_count']
        scale_results[name].append(q)

# Aggregate results
agg = {}
for name, results in scale_results.items():
    df_r = pd.DataFrame(results)
    agg[name] = {
        'avg_chunks'   : df_r['n_chunks'].mean(),
        'avg_clean_pct': df_r['both_clean'].mean(),
        'avg_tokens'   : df_r['avg_tokens'].mean(),
        'avg_cv'       : df_r['cv_tokens'].mean(),
        'total_chunks' : df_r['n_chunks'].sum(),
    }

print('\n── Aggregate Results Across All Papers ──────────────────────────────')
print(f'{"Strategy":<22} {"AvgChunks":>10} {"Clean%":>8} {"AvgTok":>8} {"AvgCV%":>8} {"TotalChunks":>12}')
print('─' * 75)
for name, a in agg.items():
    print(
        f'{name:<22} {a["avg_chunks"]:>10.1f} '
        f'{a["avg_clean_pct"]:>8.1f} '
        f'{a["avg_tokens"]:>8.1f} '
        f'{a["avg_cv"]:>8.1f} '
        f'{a["total_chunks"]:>12.0f}'
    )

## Step 9 — Chunk Size Distribution Comparison

This chart shows the distribution of chunk sizes for each strategy.  

**What you want to see:**
- A tight distribution centred near the target size (400 tokens)
- Few extreme outliers (very tiny or very large chunks)
- The sentence window strategy should be the best balance

This chart can go directly into your paper as Figure 2 or 3.

In [ ]:
# Collect token sizes for each strategy across all papers
all_token_sizes = {name: [] for name in strategy_fns}

for _, row in df.iterrows():
    text = row['full_text']
    for name, fn in strategy_fns.items():
        chunks = fn(text)
        all_token_sizes[name].extend([est_tokens(c) for c in chunks])

# Plot
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.patch.set_facecolor('#0d1117')
colors = [ACCENT, ACCENT2, ACCENT3, ACCENT4]

for ax, (name, sizes), color in zip(axes.flat, all_token_sizes.items(), colors):
    sizes_arr = np.array(sizes)

    ax.hist(sizes_arr.clip(0, 800), bins=40, color=color, alpha=0.8, edgecolor='none')
    ax.axvline(FINAL_CHUNK_SIZE, color='white', linestyle='--',
               linewidth=1.2, label=f'Target: {FINAL_CHUNK_SIZE}')
    ax.axvline(sizes_arr.mean(), color='#f97316',
               linestyle=':', linewidth=1.2,
               label=f'Mean: {sizes_arr.mean():.0f}')

    # Shade the ideal range (300–500 tokens)
    ax.axvspan(300, 500, alpha=0.07, color='white', label='Ideal range')

    ax.set_title(name, fontsize=11, pad=8, color='#e2e8f0')
    ax.set_xlabel('Chunk size (tokens)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

    # Stats annotation
    stats_text = f'n={len(sizes):,}  std={sizes_arr.std():.0f}'
    ax.text(0.97, 0.95, stats_text, transform=ax.transAxes,
            ha='right', va='top', fontsize=8, color='#64748b')

plt.suptitle('Chunk Size Distribution — Four Strategies', fontsize=13,
             y=1.01, color='#e2e8f0')
plt.tight_layout()
plt.savefig(DATA_PROC / 'chunking_size_distribution.png', dpi=150,
            bbox_inches='tight', facecolor='#0d1117')
plt.show()
print('✓ Chart saved to 1_data/processed/chunking_size_distribution.png')

## Step 10 — Overlap Effect Analysis

Now we look at a specific property of sentence window chunking — how overlap affects boundary coverage.

**Why overlap matters:**  
Imagine the answer to a query spans the boundary between chunk 3 and chunk 4.  
Without overlap, the answer is split and neither chunk retrieves the full context.  
With overlap, chunk 4 starts with the last few sentences of chunk 3 — the boundary is bridged.

**But overlap has a cost:**  
Higher overlap → more chunks → more storage and more embedding compute.  
We find the sweet spot where overlap coverage is good but chunk count is still reasonable.

In [ ]:
overlap_analysis = []

for chunk_size in CHUNK_SIZES:
    for overlap in OVERLAP_SIZES:
        if overlap >= chunk_size:      # overlap must be less than chunk size
            continue

        total_chunks   = 0
        total_clean    = 0
        total_papers   = 0

        for _, row in df.iterrows():
            chunks = chunk_sentence_window(row['full_text'], chunk_size, overlap)
            if not chunks:
                continue
            q = analyse_boundary_quality(chunks)
            total_chunks += q['n_chunks']
            total_clean  += q['both_clean'] * q['n_chunks'] / 100
            total_papers += 1

        if total_papers == 0:
            continue

        overlap_analysis.append({
            'chunk_size'      : chunk_size,
            'overlap'         : overlap,
            'overlap_pct'     : round(overlap / chunk_size * 100, 1),
            'avg_chunks_paper': round(total_chunks / total_papers, 1),
            'clean_pct'       : round(total_clean / total_chunks * 100, 1),
        })

df_overlap = pd.DataFrame(overlap_analysis)

print('── Overlap Analysis (Sentence Window Strategy) ──────────────────────')
print(f'{"ChunkSz":>8} {"Overlap":>8} {"Ovlp%":>6} {"Chunks/Paper":>13} {"Clean%":>8}')
print('─' * 50)
for _, row in df_overlap.iterrows():
    marker = ' ← selected' if row['chunk_size'] == FINAL_CHUNK_SIZE and row['overlap'] == FINAL_OVERLAP else ''
    print(
        f'{row["chunk_size"]:>8.0f} '
        f'{row["overlap"]:>8.0f} '
        f'{row["overlap_pct"]:>6.1f} '
        f'{row["avg_chunks_paper"]:>13.1f} '
        f'{row["clean_pct"]:>8.1f}'
        f'{marker}'
    )

## Step 11 — Visualise Overlap Trade-off

This heatmap shows the trade-off between chunk count (cost) and boundary quality (benefit) across all parameter combinations. The selected configuration should be in a region of high quality without excessive chunk multiplication.

In [ ]:
if len(df_overlap) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.patch.set_facecolor('#0d1117')

    # Pivot for heatmap
    pivot_clean  = df_overlap.pivot(index='overlap', columns='chunk_size', values='clean_pct')
    pivot_chunks = df_overlap.pivot(index='overlap', columns='chunk_size', values='avg_chunks_paper')

    # Heatmap 1: Clean boundary %
    ax1 = axes[0]
    sns.heatmap(
        pivot_clean, ax=ax1, cmap='Blues', annot=True, fmt='.1f',
        linewidths=0.5, linecolor='#1e293b',
        cbar_kws={'label': 'Clean boundary %'}
    )
    ax1.set_title('Boundary Quality (%)', fontsize=11, pad=10, color='#e2e8f0')
    ax1.set_xlabel('Chunk size (tokens)')
    ax1.set_ylabel('Overlap (tokens)')

    # Mark selected configuration
    if FINAL_CHUNK_SIZE in pivot_clean.columns and FINAL_OVERLAP in pivot_clean.index:
        col_idx = list(pivot_clean.columns).index(FINAL_CHUNK_SIZE)
        row_idx = list(pivot_clean.index).index(FINAL_OVERLAP)
        ax1.add_patch(mpatches.Rectangle(
            (col_idx, row_idx), 1, 1,
            fill=False, edgecolor=ACCENT3, linewidth=2.5
        ))

    # Heatmap 2: Avg chunks per paper
    ax2 = axes[1]
    sns.heatmap(
        pivot_chunks, ax=ax2, cmap='Purples', annot=True, fmt='.1f',
        linewidths=0.5, linecolor='#1e293b',
        cbar_kws={'label': 'Avg chunks per paper'}
    )
    ax2.set_title('Chunks per Paper (cost)', fontsize=11, pad=10, color='#e2e8f0')
    ax2.set_xlabel('Chunk size (tokens)')
    ax2.set_ylabel('Overlap (tokens)')

    if FINAL_CHUNK_SIZE in pivot_chunks.columns and FINAL_OVERLAP in pivot_chunks.index:
        col_idx = list(pivot_chunks.columns).index(FINAL_CHUNK_SIZE)
        row_idx = list(pivot_chunks.index).index(FINAL_OVERLAP)
        ax2.add_patch(mpatches.Rectangle(
            (col_idx, row_idx), 1, 1,
            fill=False, edgecolor=ACCENT3, linewidth=2.5
        ))

    plt.suptitle(
        f'Chunking Parameter Analysis — Sentence Window Strategy\n'
        f'(green box = selected: {FINAL_CHUNK_SIZE} tok, {FINAL_OVERLAP} overlap)',
        fontsize=12, y=1.02, color='#e2e8f0'
    )
    plt.tight_layout()
    plt.savefig(DATA_PROC / 'chunking_parameter_heatmap.png', dpi=150,
                bbox_inches='tight', facecolor='#0d1117')
    plt.show()
    print('✓ Heatmap saved to 1_data/processed/chunking_parameter_heatmap.png')

## Step 12 — Strategy Selection Justification

This cell documents your decision for the paper. Every design decision in a research paper needs justification with evidence — not just `"we chose X"` but `"we chose X because the evidence showed Y"`.

In [ ]:
print('═' * 65)
print('  CHUNKING STRATEGY SELECTION — JUSTIFICATION')
print('═' * 65)
print()
print('Selected: Sentence Window Chunking')
print(f'  Chunk size : {FINAL_CHUNK_SIZE} tokens')
print(f'  Overlap    : {FINAL_OVERLAP} tokens ({FINAL_OVERLAP/FINAL_CHUNK_SIZE*100:.0f}% of chunk)')
print()
print('Evidence from this notebook:')

sw_agg = agg.get('Sentence Window', {})
fc_agg = agg.get('Fixed Character', {})
ft_agg = agg.get('Fixed Token', {})
pg_agg = agg.get('Paragraph', {})

print()
print('Boundary quality (Clean %):')
for name, a in agg.items():
    marker = ' ← selected' if name == 'Sentence Window' else ''
    print(f'  {name:<22}: {a["avg_clean_pct"]:.1f}%{marker}')

print()
print('Consistency (lower CV = more uniform sizes):')
for name, a in agg.items():
    marker = ' ← selected' if name == 'Sentence Window' else ''
    print(f'  {name:<22}: CV = {a["avg_cv"]:.1f}%{marker}')

print()
print('Rationale:')
print('  1. Sentence Window produces the cleanest boundaries')
print('     (never cuts mid-sentence unlike Fixed Character/Token)')
print('  2. More consistent chunk sizes than Paragraph strategy')
print('     (Paragraph produces huge variance when abstracts have no breaks)')
print('  3. 400-token target fits within BGE-M3 context window')
print('     with room for the title prepend (Title: ... Content: ...)')
print('  4. 50-token overlap (12.5%) bridges sentence boundaries')
print('     without excessive chunk multiplication')
print()
print('Paper citation text (paste into Section 3.2):')
print('  "We segment each paper into chunks using a sentence-window')
print('  strategy with a target size of 400 tokens and 50-token overlap.')
print('  This strategy preserves sentence boundaries, ensuring embedding')
print('  vectors are not diluted by mid-sentence splits. We evaluated')
print('  four strategies on 100 papers and selected sentence windowing')
print('  based on boundary quality and size consistency metrics."')
print('═' * 65)

## Step 13 — Build Final Chunk Dataset and Save

Apply the selected strategy to all papers and save for Notebook 03.

In [ ]:
# Reload full paper dataset (not just the 100-paper sample)
df_all = pd.read_parquet(papers_path)
df_all['full_text'] = df_all['title_clean'].fillna('') + '. ' + df_all['abstract_clean'].fillna('')

print(f'Building final chunk dataset from {len(df_all):,} papers...')

records = []
for _, row in tqdm(df_all.iterrows(), total=len(df_all), desc='Chunking'):
    chunks = chunk_sentence_window(
        row['full_text'],
        FINAL_CHUNK_SIZE,
        FINAL_OVERLAP
    )
    for idx, chunk in enumerate(chunks):
        records.append({
            'chunk_id'          : f"{row['cord_uid']}_c{idx:03d}",
            'cord_uid'          : row['cord_uid'],
            'title'             : row['title_clean'],
            'chunk_text'        : chunk,
            'chunk_index'       : idx,
            'chunk_tokens'      : est_tokens(chunk),
            # Prepend title for BGE-M3 — standard best practice
            'text_for_embedding': f"Title: {row['title_clean']}. Content: {chunk}",
            'publish_time'      : row.get('publish_time', ''),
            'journal'           : row.get('journal', ''),
            'url'               : row.get('url', ''),
        })

df_chunks = pd.DataFrame(records)

# Save
chunks_path = DATA_PROC / 'chunks.parquet'
df_chunks.to_parquet(chunks_path, index=False)

# Save chunking config
chunking_config = {
    'strategy'     : 'sentence_window',
    'chunk_tokens' : FINAL_CHUNK_SIZE,
    'overlap_tokens': FINAL_OVERLAP,
    'title_prepend': True,
    'n_papers'     : len(df_all),
    'n_chunks'     : len(df_chunks),
    'avg_chunks_per_paper': round(len(df_chunks) / len(df_all), 2),
    'avg_chunk_tokens': round(df_chunks['chunk_tokens'].mean(), 1),
}
with open(DATA_PROC / 'chunking_config.json', 'w') as f:
    json.dump(chunking_config, f, indent=2)

print(f'\n✓ Chunks saved to {chunks_path}')
print(f'  Papers  : {len(df_all):,}')
print(f'  Chunks  : {len(df_chunks):,}')
print(f'  Avg chunks/paper : {len(df_chunks)/len(df_all):.1f}')
print(f'  Avg chunk tokens : {df_chunks["chunk_tokens"].mean():.0f}')
print(f'  File size        : {chunks_path.stat().st_size / 1e6:.1f} MB')

## Step 14 — Summary

In [ ]:
print('═' * 60)
print('  Notebook 02 — Text Chunking — COMPLETE')
print('═' * 60)
print()
print('What we did:')
print('  ✓ Implemented 4 chunking strategies from scratch')
print('  ✓ Compared boundary quality across 100 papers')
print('  ✓ Analysed chunk size distributions with charts')
print('  ✓ Tested overlap parameters (4 sizes × 4 overlaps)')
print('  ✓ Selected sentence window (400 tok, 50 overlap)')
print('  ✓ Built and saved final chunk dataset')
print()
print('Files saved:')
print(f'  1_data/processed/chunks.parquet')
print(f'  1_data/processed/chunking_config.json')
print(f'  1_data/processed/chunking_size_distribution.png')
print(f'  1_data/processed/chunking_parameter_heatmap.png')
print()
print('Next — Notebook 03: Embedding Baseline')
print('  → Load chunks.parquet')
print('  → Run BGE-M3 on all chunks')
print('  → Understand the embedding space (visualise with UMAP)')
print('  → Compare BGE-M3 vs the 2022 DPR embeddings')
print('  → Save embeddings for Notebook 04')
print('═' * 60)